In [1]:
import os, time, statistics
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation
from tqdm.notebook import tqdm


DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CLASSES       = 11
IMG_H           = 640    
IMG_W           = 640
BATCH_SIZE      = 2        
ACCUM_STEPS     = 4       
N_EPOCHS        = 20
WARMUP_EPOCHS   = 3        
LR              = 6e-5
WEIGHT_DECAY    = 1e-4
FOCAL_GAMMA     = 2.0

# Desert dataset color statistics (swap for computed values if you have them)
MEAN = [0.452, 0.431, 0.376]
STD  = [0.218, 0.213, 0.207]

# Raw pixel value → class index
VALUE_MAP = {
    0:     0,   # Background
    100:   1,   # Trees
    200:   2,   # Lush Bushes
    300:   3,   # Dry Grass
    500:   4,   # Dry Bushes
    550:   5,   # Ground Clutter
    600:   6,   # Flowers
    700:   7,   # Logs
    800:   8,   # Rocks
    7100:  9,   # Landscape
    10000: 10,  # Sky
}

# Update these to your actual paths
TRAIN_DIR  = './Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train'
VAL_DIR    = './Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val'
OUTPUT_DIR = './train_stats'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device : cuda
GPU    : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM   : 6.4 GB


In [2]:
def remap_mask(mask_bgr: np.ndarray) -> np.ndarray:
    """
    Convert a raw BGR mask image into a single-channel uint8 class-index mask.

    Raw masks store their value across two channels:
        raw_value = B_channel + G_channel * 256
    This handles values up to 65535, comfortably covering 10000 (Sky).
    """
    raw = (mask_bgr[:, :, 0].astype(np.int32)
           + mask_bgr[:, :, 1].astype(np.int32) * 256)

    out = np.zeros(raw.shape, dtype=np.uint8)          # default → class 0
    for raw_val, cls_idx in VALUE_MAP.items():
        out[raw == raw_val] = cls_idx
    return out


# Quick sanity check
dummy_bgr = np.zeros((4, 4, 3), dtype=np.uint8)
dummy_bgr[0, 0] = [100, 0, 0]    # raw=100  → class 1 (Trees)
dummy_bgr[1, 0] = [16, 39, 0]    # raw=16 + 39*256 = 10000 → class 9 (Sky)
result = remap_mask(dummy_bgr)
assert result[0, 0] == 1, 'Trees mapping failed'
assert result[1, 0] == 10, 'Sky mapping failed'
print('remap_mask ✓  (Trees=1, Sky=10 verified)')

remap_mask ✓  (Trees=1, Sky=10 verified)


## Cell 4 — Albumentations Transforms

In [3]:
def build_train_transform() -> A.Compose:
    """Aggressive augmentation for desert/offroad UGV imagery."""
    return A.Compose([
        # --- Spatial ---
        A.Resize(IMG_H, IMG_W, interpolation=cv2.INTER_LINEAR),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101, p=0.4
        ),

        # --- Photometric: desert lighting ---
        A.OneOf([
            A.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.2, hue=0.05, p=1.0),
            A.RandomBrightnessContrast(
                brightness_limit=0.3, contrast_limit=0.3, p=1.0),
        ], p=0.7),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.4),

        # --- Sensor / atmospheric noise ---
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, p=0.15),

        # --- Motion blur (moving vehicle) ---
        A.OneOf([
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.2),

        # --- Normalize & convert ---
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def build_val_transform() -> A.Compose:
    """Deterministic: resize + normalize only."""
    return A.Compose([
        A.Resize(IMG_H, IMG_W, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


print('Transforms defined ✓')
print(f'  Train: {len(build_train_transform().transforms)} transform stages')
print(f'  Val  : {len(build_val_transform().transforms)} transform stages')

Transforms defined ✓
  Train: 11 transform stages
  Val  : 3 transform stages


/home/akshay/miniconda3/envs/alphawave/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_2113/2614311116.py:23: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
/tmp/ipykernel_2113/2614311116.py:25: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, p=0.15),


In [4]:
class OffroadSegDataset(Dataset):
    """
    Expects:
        data_dir/
            Color_Images/
            Segmentation/
    """

    def __init__(self, data_dir: str, transform: A.Compose = None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.mask_dir  = os.path.join(data_dir, 'Segmentation')
        self.transform = transform

        self.ids = sorted([
            f for f in os.listdir(self.image_dir)
            if f.endswith('.png')
        ])

        assert len(self.ids) > 0, f'No .png images found in {self.image_dir}'

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        # ---- Load image ----
        img = cv2.imread(os.path.join(self.image_dir, name), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ---- Load mask WITHOUT losing information ----
        mask_path = os.path.join(self.mask_dir, name)
        mask_raw = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

        # Convert to raw label values
        if mask_raw.ndim == 3:
            raw = mask_raw[:,:,0].astype(np.int32) + mask_raw[:,:,1].astype(np.int32)*256
        else:
            raw = mask_raw.astype(np.int32)

        # Map raw values → class indices
        mask = np.zeros(raw.shape, dtype=np.uint8)
        for raw_val, cls_idx in VALUE_MAP.items():
            mask[raw == raw_val] = cls_idx

        if self.transform:
            out = self.transform(image=img, mask=mask)
            img = out['image']
            mask = out['mask']

        return img, mask.long()


# ── Instantiate datasets & loaders ────────────────────────────────────────
train_ds = OffroadSegDataset(TRAIN_DIR, transform=build_train_transform())
val_ds   = OffroadSegDataset(VAL_DIR,   transform=build_val_transform())

print("Checking TRAIN masks")
for i in range(5):
    _, m = train_ds[i]
    print(f"train sample {i} classes:", torch.unique(m))

print("\nChecking VAL masks")
for i in range(5):
    _, m = val_ds[i]
    print(f"val sample {i} classes:", torch.unique(m))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train samples : {len(train_ds)}')
print(f'Val samples   : {len(val_ds)}')

# Verify one batch
imgs, masks = next(iter(train_loader))
print(f'Batch image shape : {imgs.shape}   dtype={imgs.dtype}')
print(f'Batch mask shape  : {masks.shape}  dtype={masks.dtype}')
print(f'Classes in batch  : {masks.unique().tolist()}')

Checking TRAIN masks


/tmp/ipykernel_2113/2614311116.py:23: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
/tmp/ipykernel_2113/2614311116.py:25: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, p=0.15),


train sample 0 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
train sample 1 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
train sample 2 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
train sample 3 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
train sample 4 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])

Checking VAL masks
val sample 0 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
val sample 1 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
val sample 2 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
val sample 3 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
val sample 4 classes: tensor([ 2,  3,  4,  5,  8,  9, 10])
Train samples : 2857
Val samples   : 317
Batch image shape : torch.Size([2, 3, 640, 640])   dtype=torch.float32
Batch mask shape  : torch.Size([2, 640, 640])  dtype=torch.int64
Classes in batch  : [1, 2, 3, 5, 7, 8, 9, 10]


## Cell 6 — SegFormer Model

In [5]:
# SegFormer-B0: fastest & smallest — ideal for 6GB VRAM + <50ms target
# ignore_mismatched_sizes discards the 150-class ADE20K head
# and randomly initialises a fresh 10-class MLP decoder
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/mit-b2',
    num_labels=N_CLASSES,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

# ── Staged freezing: freeze MiT encoder for warm-up phase ─────────────────
def freeze_encoder(m):
    for p in m.segformer.parameters():
        p.requires_grad = False
    print('Encoder frozen  (decoder-only warm-up)')

def unfreeze_encoder(m):
    for p in m.segformer.parameters():
        p.requires_grad = True
    print('Encoder unfrozen (end-to-end fine-tuning)')

freeze_encoder(model)

total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable now   : {trainable:,}  (decoder only during warm-up)')

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Encoder frozen  (decoder-only warm-up)
Total params    : 27,355,083
Trainable now   : 3,158,795  (decoder only during warm-up)


## Cell 7 — Focal + Dice Loss

In [6]:
class FocalLoss(nn.Module):
    """Multi-class Focal Loss — focuses on hard misclassified pixels."""
    def __init__(self, gamma=2.0, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        log_p  = F.log_softmax(logits, dim=1)
        ce     = F.nll_loss(log_p, targets,
                            ignore_index=self.ignore_index, reduction='none')
        p_t    = torch.exp(-ce)
        loss   = ((1 - p_t) ** self.gamma) * ce
        valid  = targets != self.ignore_index
        return loss[valid].mean()


class DiceLoss(nn.Module):
    """Soft Dice Loss — maximises IoU for small rare classes (logs, rocks)."""
    def __init__(self, smooth=1.0, ignore_index=255):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        n_cls  = logits.shape[1]
        probs  = F.softmax(logits, dim=1)               # (B, C, H, W)

        valid  = (targets != self.ignore_index).unsqueeze(1)
        t_clamp = targets.clone()
        t_clamp[targets == self.ignore_index] = 0

        oh = F.one_hot(t_clamp, n_cls).permute(0, 3, 1, 2).float() * valid
        probs = probs * valid

        dims  = (0, 2, 3)
        inter = (probs * oh).sum(dim=dims)
        card  = (probs + oh).sum(dim=dims)
        dice  = (2.0 * inter + self.smooth) / (card + self.smooth)
        return 1.0 - dice.mean()


class FocalDiceLoss(nn.Module):
    """Focal + Dice combined loss."""
    def __init__(self, gamma=2.0, ignore_index=255):
        super().__init__()
        self.focal = FocalLoss(gamma=gamma, ignore_index=ignore_index)
        self.dice  = DiceLoss(ignore_index=ignore_index)

    def forward(self, logits, targets):
        return self.focal(logits, targets) + self.dice(logits, targets)


criterion = FocalDiceLoss(gamma=FOCAL_GAMMA).to(DEVICE)

# Quick test
with torch.no_grad():
    dummy_logits = torch.randn(2, N_CLASSES, IMG_H, IMG_W, device=DEVICE)
    dummy_masks  = torch.randint(0, N_CLASSES, (2, IMG_H, IMG_W), device=DEVICE)
    test_loss = criterion(dummy_logits, dummy_masks)
print(f'FocalDiceLoss ✓  (test loss = {test_loss.item():.4f})')

FocalDiceLoss ✓  (test loss = 3.3996)


## Cell 8 — Optimizer, Scheduler & AMP Scaler

In [7]:
# AdamW — Adam with decoupled weight decay (prevents transformer weights inflating)
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing — smoothly reduces LR toward 0, avoids overshooting
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

# AMP GradScaler — scales gradients to prevent FP16 underflow
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

print('Optimizer  : AdamW')
print('Scheduler  : CosineAnnealingLR')
print(f'AMP scaler : enabled={DEVICE.type == "cuda"}')

Optimizer  : AdamW
Scheduler  : CosineAnnealingLR
AMP scaler : enabled=True


/tmp/ipykernel_2113/1277032324.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


## Cell 9 — Metrics Helper

In [8]:
def compute_metrics(logits: torch.Tensor,
                    targets: torch.Tensor,
                    n_classes: int = N_CLASSES,
                    smooth: float = 1e-6):
    """
    Returns (mean_iou, mean_dice, pixel_accuracy) as Python floats.
    NaN classes (not present in batch) are excluded from the mean.
    """
    preds = logits.argmax(dim=1).view(-1)
    tgts  = targets.view(-1)

    iou_list, dice_list = [], []
    for c in range(n_classes):
        p = preds == c
        t = tgts  == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        iou_list.append(
            (inter / (union + smooth)).item() if union > 0 else float('nan')
        )
        denom = p.sum().float() + t.sum().float()
        dice_list.append(((2 * inter + smooth) / (denom + smooth)).item())

    pixel_acc = (preds == tgts).float().mean().item()
    return float(np.nanmean(iou_list)), float(np.mean(dice_list)), pixel_acc


print('compute_metrics ✓')

compute_metrics ✓


## Cell 10 — Training & Validation Loop Functions

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, scaler,
                    device, accum_steps):
    model.train()
    losses, ious, dices, accs = [], [], [], []
    optimizer.zero_grad()

    pbar = tqdm(loader, desc='  Train', leave=False)
    for step, (imgs, masks) in enumerate(pbar, 1):
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        # ── Phase 4.1: Forward pass under AMP ─────────────────────────────
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            raw_logits = model(pixel_values=imgs).logits   # (B, C, H/4, W/4)

            # ── Phase 4.2: Upsample from 1/4 scale back to full resolution
            logits = F.interpolate(
                raw_logits,
                size=(IMG_H, IMG_W),
                mode='bilinear',
                align_corners=False
            )                                              # (B, C, H, W)

            # ── Phase 4.3: Loss & gradient accumulation ────────────────────
            loss = criterion(logits, masks) / accum_steps

        scaler.scale(loss).backward()

        # ── Phase 4.4: Optimizer step every accum_steps batches ───────────
        if step % accum_steps == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            iou, dice, acc = compute_metrics(logits.detach(), masks)

        losses.append(loss.item() * accum_steps)
        ious.append(iou); dices.append(dice); accs.append(acc)
        pbar.set_postfix(loss=f'{np.mean(losses):.3f}', iou=f'{np.nanmean(ious):.3f}')

    return (float(np.mean(losses)), float(np.nanmean(ious)),
            float(np.mean(dices)),  float(np.mean(accs)))


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    losses, ious, dices, accs = [], [], [], []

    pbar = tqdm(loader, desc='    Val', leave=False)
    for imgs, masks in pbar:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            raw_logits = model(pixel_values=imgs).logits
            logits = F.interpolate(
                raw_logits, size=(IMG_H, IMG_W),
                mode='bilinear', align_corners=False
            )
            loss = criterion(logits, masks)

        iou, dice, acc = compute_metrics(logits, masks)
        losses.append(loss.item())
        ious.append(iou); dices.append(dice); accs.append(acc)
        pbar.set_postfix(iou=f'{np.nanmean(ious):.3f}')

    return (float(np.mean(losses)), float(np.nanmean(ious)),
            float(np.mean(dices)),  float(np.mean(accs)))


print('train_one_epoch & validate defined ✓')

train_one_epoch & validate defined ✓


## Cell 11 — Main Training Loop

In [10]:
CKPT_PATH = "best_model_final.pth"

history = {k: [] for k in [
    "train_loss","val_loss",
    "train_iou","val_iou",
    "train_dice","val_dice",
    "train_acc","val_acc"
]}

start_epoch = 1
best_iou = 0.0


# ── Resume training if checkpoint exists ─────────────────────────────
if os.path.exists(CKPT_PATH):

    print("Checkpoint found. Loading...")

    checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_iou = checkpoint["val_iou"]

    print(f"Resuming from epoch {start_epoch}")
    print(f"Best IoU so far: {best_iou:.4f}")


print("=" * 65)
print("Starting training")
print(f"  Epochs          : {N_EPOCHS}")
print(f"  Warm-up epochs  : {WARMUP_EPOCHS}")
print(f"  Batch size      : {BATCH_SIZE}  (effective {BATCH_SIZE*ACCUM_STEPS})")
print("=" * 65)


for epoch in range(start_epoch, N_EPOCHS + 1):


    # ── Encoder unfreeze after warm-up ───────────────────────────────
    if epoch == max(start_epoch, WARMUP_EPOCHS + 1):

        unfreeze_encoder(model)

        optimizer = optim.AdamW([
            {"params": model.segformer.parameters(), "lr": LR * 0.1},
            {"params": model.decode_head.parameters(), "lr": LR}
        ], weight_decay=WEIGHT_DECAY)

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=N_EPOCHS - WARMUP_EPOCHS,
            eta_min=1e-6
        )

        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


    print(f"\nEpoch {epoch}/{N_EPOCHS}")


    tr_loss, tr_iou, tr_dice, tr_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        DEVICE,
        ACCUM_STEPS
    )


    vl_loss, vl_iou, vl_dice, vl_acc = validate(
        model,
        val_loader,
        criterion,
        DEVICE
    )


    scheduler.step()


    # ── Record metrics ───────────────────────────────────────────────
    for k, v in zip(
        history.keys(),
        [tr_loss, vl_loss, tr_iou, vl_iou, tr_dice, vl_dice, tr_acc, vl_acc]
    ):
        history[k].append(v)


    print(f"  Train  loss={tr_loss:.4f}  IoU={tr_iou:.4f}  Dice={tr_dice:.4f}  Acc={tr_acc:.4f}")
    print(f"  Val    loss={vl_loss:.4f}  IoU={vl_iou:.4f}  Dice={vl_dice:.4f}  Acc={vl_acc:.4f}")


    # ── Save best checkpoint ─────────────────────────────────────────
    if vl_iou > best_iou:

        best_iou = vl_iou

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_iou": vl_iou
        }, CKPT_PATH)

        print(f"✓ New best IoU {best_iou:.4f} — saved to {CKPT_PATH}")


print(f"\nTraining complete. Best Val IoU: {best_iou:.4f}")

Starting training
  Epochs          : 20
  Warm-up epochs  : 3
  Batch size      : 2  (effective 8)

Epoch 1/20


  Train:   0%|          | 0/1428 [00:00<?, ?it/s]

/tmp/ipykernel_2113/455551485.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


    Val:   0%|          | 0/159 [00:00<?, ?it/s]

/tmp/ipykernel_2113/455551485.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


...

    Val:   0%|          | 0/159 [00:00<?, ?it/s]

  Train  loss=0.6913  IoU=0.4986  Dice=0.6827  Acc=0.8366
  Val    loss=0.5974  IoU=0.5617  Dice=0.7626  Acc=0.8705
✓ New best IoU 0.5617 — saved to best_model_final.pth

Epoch 20/20


  Train:   0%|          | 0/1428 [00:00<?, ?it/s]

    Val:   0%|          | 0/159 [00:00<?, ?it/s]

  Train  loss=0.6916  IoU=0.4986  Dice=0.6814  Acc=0.8365
  Val    loss=0.5961  IoU=0.5633  Dice=0.7646  Acc=0.8704
✓ New best IoU 0.5633 — saved to best_model_final.pth

Training complete. Best Val IoU: 0.5633


## Cell 12 — Training Plots

In [11]:
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('SegFormer Training — Offroad Segmentation', fontsize=14)

pairs = [
    ('train_loss',  'val_loss',  'Loss',           axes[0, 0]),
    ('train_iou',   'val_iou',   'Mean IoU',       axes[0, 1]),
    ('train_dice',  'val_dice',  'Dice Score',     axes[1, 0]),
    ('train_acc',   'val_acc',   'Pixel Accuracy', axes[1, 1]),
]
for tr_key, vl_key, title, ax in pairs:
    ax.plot(epochs, history[tr_key], label='train', marker='o', markersize=3)
    ax.plot(epochs, history[vl_key], label='val',   marker='s', markersize=3)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'all_metrics3.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to: {plot_path}')

Plot saved to: ./train_stats/all_metrics3.png


/tmp/ipykernel_2113/1348063390.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cell 13 — Save Metrics to Text File

In [12]:
metrics_path = os.path.join(OUTPUT_DIR, 'metrics3.txt')
n = len(history['train_loss'])

with open(metrics_path, 'w') as f:
    f.write('SEGFORMER TRAINING RESULTS\n' + '=' * 65 + '\n\n')
    f.write(f'Best Val IoU  : {best_iou:.4f}\n')
    f.write(f'Best Val Dice : {max(history["val_dice"]):.4f}  '
            f'(Epoch {history["val_dice"].index(max(history["val_dice"]))+1})\n')
    f.write(f'Best Val Acc  : {max(history["val_acc"]):.4f}  '
            f'(Epoch {history["val_acc"].index(max(history["val_acc"]))+1})\n')
    f.write(f'Lowest Val Loss: {min(history["val_loss"]):.4f}  '
            f'(Epoch {history["val_loss"].index(min(history["val_loss"]))+1})\n\n')

    header = f"{'Ep':>3} {'TrLoss':>8} {'VLoss':>8} {'TrIoU':>7} {'VIoU':>7} "\
             f"{'TrDice':>7} {'VDice':>7} {'TrAcc':>7} {'VAcc':>7}\n"
    f.write(header)
    f.write('-' * len(header) + '\n')
    for i in range(n):
        f.write(f"{i+1:>3} "
                f"{history['train_loss'][i]:>8.4f} {history['val_loss'][i]:>8.4f} "
                f"{history['train_iou'][i]:>7.4f} {history['val_iou'][i]:>7.4f} "
                f"{history['train_dice'][i]:>7.4f} {history['val_dice'][i]:>7.4f} "
                f"{history['train_acc'][i]:>7.4f} {history['val_acc'][i]:>7.4f}\n")

print(f'Metrics saved to: {metrics_path}')

Metrics saved to: ./train_stats/metrics3.txt


## Cell 14 — Phase 5.2: Inference Benchmark (<50ms)

In [13]:
TARGET_MS      = 50.0
WARMUP_RUNS    = 10
BENCHMARK_RUNS = 50

# ── Load best checkpoint ───────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (Val IoU={ckpt['val_iou']:.4f})")

# ── Dummy input — exact inference dimensions ───────────────────────────────
dummy = torch.randn(1, 3, IMG_H, IMG_W, device=DEVICE)

# ── Warm-up (10 passes) ────────────────────────────────────────────────────
print(f'Warming up ({WARMUP_RUNS} passes)…')
with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = model(pixel_values=dummy).logits
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

# ── 50-pass timed benchmark ────────────────────────────────────────────────
print(f'Benchmarking ({BENCHMARK_RUNS} passes)…')
latencies = []

if DEVICE.type == 'cuda':
    # CUDA Events give microsecond GPU-side accuracy
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    with torch.no_grad():
        for _ in range(BENCHMARK_RUNS):
            start_ev.record()
            _ = model(pixel_values=dummy).logits
            end_ev.record()
            torch.cuda.synchronize()
            latencies.append(start_ev.elapsed_time(end_ev))
else:
    with torch.no_grad():
        for _ in range(BENCHMARK_RUNS):
            t0 = time.perf_counter()
            _ = model(pixel_values=dummy).logits
            latencies.append((time.perf_counter() - t0) * 1000)

mean_ms = statistics.mean(latencies)
std_ms  = statistics.stdev(latencies)
p90_ms  = sorted(latencies)[int(0.90 * BENCHMARK_RUNS)]

print(f'\n── Latency Report ─────────────────────────────────')
print(f'  Mean : {mean_ms:.2f} ms')
print(f'  Std  : {std_ms:.2f} ms')
print(f'  Min  : {min(latencies):.2f} ms')
print(f'  Max  : {max(latencies):.2f} ms')
print(f'  P90  : {p90_ms:.2f} ms')
print(f'  Target: {TARGET_MS} ms')

if mean_ms <= TARGET_MS:
    print(f'\n  ✓ PASS — {mean_ms:.2f}ms ≤ {TARGET_MS}ms  →  Ready for submission!')
else:
    print(f'\n  ✗ FAIL — {mean_ms:.2f}ms > {TARGET_MS}ms  →  Run Cell 15 to optimise.')

Loaded checkpoint from epoch 20 (Val IoU=0.5633)
Warming up (10 passes)…
Benchmarking (50 passes)…

── Latency Report ─────────────────────────────────
  Mean : 67.45 ms
  Std  : 5.22 ms
  Min  : 62.37 ms
  Max  : 92.52 ms
  P90  : 73.58 ms
  Target: 50.0 ms

  ✗ FAIL — 67.45ms > 50.0ms  →  Run Cell 15 to optimise.


In [14]:
import torch
import numpy as np
import cv2
import os
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "best_model_final.pth"
DATA_DIR = "./Offroad_Segmentation_testImages/Offroad_Segmentation_testImages"
OUTPUT_DIR = "./predictions1"

IMG_H = 640
IMG_W = 640
N_CLASSES = 11

os.makedirs(OUTPUT_DIR, exist_ok=True)

VALUE_MAP = {
0:0,
100:1,
200:2,
300:3,
500:4,
550:5,
600:6,
700:7,
800:8,
7100:9,
10000:10
}

def remap_mask(mask):

    if len(mask.shape) == 2:
        raw = mask.astype(np.int32)
    else:
        raw = mask[:,:,0].astype(np.int32) + mask[:,:,1].astype(np.int32)*256

    mapped = np.zeros(raw.shape, dtype=np.uint8)

    for raw_val, cls in VALUE_MAP.items():
        mapped[raw == raw_val] = cls

    return mapped

def compute_iou(pred, gt):

    pred = pred.flatten()
    gt = gt.flatten()

    ious = []

    for c in range(N_CLASSES):

        p = pred == c
        t = gt == c

        inter = np.logical_and(p,t).sum()
        union = np.logical_or(p,t).sum()

        if union == 0:
            continue

        ious.append(inter/union)

    if len(ious)==0:
        return 0

    return np.mean(ious)

transform = A.Compose([
A.Resize(IMG_H, IMG_W),
A.Normalize(mean=[0.452,0.431,0.376],
std=[0.218,0.213,0.207]),
ToTensorV2()
])

print("Loading model...")

model = SegformerForSemanticSegmentation.from_pretrained(
"nvidia/mit-b2",
num_labels=N_CLASSES,
ignore_mismatched_sizes=True
)

checkpoint = torch.load(MODEL_PATH,map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(DEVICE)
model.eval()

print("Model loaded")

image_dir = os.path.join(DATA_DIR,"Color_Images")
mask_dir = os.path.join(DATA_DIR,"Segmentation")

ids = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])

print("Found",len(ids),"images")

iou_scores = []
debug_printed = False

with torch.no_grad():

    for name in tqdm(ids):

        img_path = os.path.join(image_dir,name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        data = transform(image=img)
        img_tensor = data["image"].unsqueeze(0).to(DEVICE)

        logits = model(pixel_values=img_tensor).logits

        logits = F.interpolate(
            logits,
            size=(IMG_H,IMG_W),
            mode="bilinear",
            align_corners=False
        )

        pred = torch.argmax(logits,dim=1).squeeze().cpu().numpy().astype(np.uint8)

        mask_path = os.path.join(mask_dir,name)

        if os.path.exists(mask_path):

            mask_raw = cv2.imread(mask_path,cv2.IMREAD_UNCHANGED)

            gt = remap_mask(mask_raw)

            gt = cv2.resize(
                gt,
                (IMG_W,IMG_H),
                interpolation=cv2.INTER_NEAREST
            )

            # DEBUG PRINT (only once)
            if not debug_printed:
                print("\nDEBUG CHECK")
                print("Pred classes:", np.unique(pred))
                print("GT classes:", np.unique(gt))
                debug_printed = True

            iou = compute_iou(pred,gt)

            iou_scores.append(iou)

if len(iou_scores)>0:

    print("\nMean IoU:",np.mean(iou_scores))

else:

    print("\nNo IoU computed")

print("\nDone")

Loading model...


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded
Found 1002 images


  0%|          | 1/1002 [00:00<05:57,  2.80it/s]


DEBUG CHECK
Pred classes: [ 1  2  3  4  5  8  9 10]
GT classes: [ 1  2  3  4  8  9 10]


100%|██████████| 1002/1002 [02:19<00:00,  7.20it/s]


Mean IoU: 0.3357842475966141

Done
